# Reentrenamiento Cuántico — Quantum Retrain (h5)

Este notebook reentrena paso a paso el pipeline cuántico (Scaler → PCA → Quantum Kernel → SVC) para una empresa seleccionada.

## 1. Configuración

In [ ]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

from bvg_core.config import (
    COMPANY_FILE_MAP,
    DATA_PATH,
    GLOBAL_SEED,
    QUANTUM_DIR,
    QUANTUM_RETRAINED_DIR,
    TARGET_H,
)
from bvg_core.data import load_master_dataset
from bvg_core.dataset import get_latest_dataset_version
from bvg_core.features import build_features_for_company
from bvg_core.quantum import build_qkernel
from bvg_core.splits import temporal_split_company
from bvg_core.utils import build_dataset_version, load_manifest, sha256_file

# Empresa a reentrenar. Cambiar a CORPORACION_FAVORITA_CA para la otra empresa.
EMPRESA = "BANCO GUAYAQUIL S.A."
# EMPRESA = "CORPORACION FAVORITA C.A."

MODEL_NAME = TARGET_H  # "h5"
tag = COMPANY_FILE_MAP[EMPRESA]

manifest_path = QUANTUM_DIR / f"{tag}_{MODEL_NAME}_manifest.json"
manifest = load_manifest(manifest_path)

# kernel_config se guarda como artefacto separado junto al manifest
kernel_config_path = QUANTUM_DIR / f"{tag}_{MODEL_NAME}_kernel_config.json"
kernel_config = json.loads(kernel_config_path.read_text(encoding="utf-8"))

# Cargar el dataset versionado más reciente; fallback al path canónico
latest_versioned_path = get_latest_dataset_version()
if latest_versioned_path is not None:
    dataset_path = latest_versioned_path
    dataset_version = build_dataset_version(dataset_path)
else:
    dataset_path = DATA_PATH
    dataset_version = "legacy"

master_df = load_master_dataset(dataset_path)

print(f"Dataset cargado: {dataset_path}")
print(f"Versión del dataset: {dataset_version}")
print(f"Manifest cargado: {manifest_path}")
print(f"Kernel config: {kernel_config}")

## 2. Preparación de datos

In [ ]:
feature_columns = list(manifest["feature_columns"])
target_col = manifest.get("target_column", "target_up_h5")

company_df = master_df.loc[master_df["empresa"] == EMPRESA].copy()
featured = build_features_for_company(company_df)
featured = featured.dropna(subset=[target_col] + feature_columns).copy()

tr, te, X_train, y_train, X_test, y_test = temporal_split_company(
    featured,
    EMPRESA,
    test_size=30,
    feature_cols=feature_columns,
    target_col=target_col,
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

## 3. Reentrenamiento — Scaler + PCA

In [ ]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=5, random_state=GLOBAL_SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

cumulative_var = np.cumsum(pca.explained_variance_ratio_)
print("Varianza explicada acumulada:")
for i, v in enumerate(cumulative_var, start=1):
    print(f"  PC{i}: {v:.6f}")

## 4. Reentrenamiento — Kernel cuántico

In [ ]:
qkernel = build_qkernel(kernel_config)

# Matriz de kernel de entrenamiento
K_train = qkernel.evaluate(x_vec=X_train_pca)

print(f"K_train shape: {K_train.shape}")
print(f"K_train min={K_train.min():.6f} max={K_train.max():.6f} mean={K_train.mean():.6f}")

## 5. Reentrenamiento — SVC

In [ ]:
svc = SVC(
    kernel="precomputed",
    probability=True,
    class_weight="balanced",
    random_state=GLOBAL_SEED,
)
svc.fit(K_train, y_train)

print(f"Número de vectores de soporte: {svc.n_support_.sum()}")

## 6. Smoke test

In [ ]:
# Matriz de kernel de test
K_test = qkernel.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)
y_pred = svc.predict(K_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test F1: {f1:.4f}")

if accuracy <= 0.3:
    raise ValueError(f"Smoke test falló: accuracy={accuracy:.3f} en test (<=0.3). NO se guardarán artefactos.")

## 7. Guardar artefactos

In [ ]:
date_str = datetime.now(timezone.utc).strftime("%Y%m%d")
model_version = f"retrained_{date_str}"

out_dir = QUANTUM_RETRAINED_DIR / date_str
out_dir.mkdir(parents=True, exist_ok=True)

scaler_path = out_dir / f"{tag}_{MODEL_NAME}_scaler_retrained_{date_str}.joblib"
pca_path = out_dir / f"{tag}_{MODEL_NAME}_pca_retrained_{date_str}.joblib"
svc_path = out_dir / f"{tag}_{MODEL_NAME}_svc_retrained_{date_str}.joblib"
kernel_config_out_path = out_dir / f"{tag}_{MODEL_NAME}_kernel_config_retrained_{date_str}.json"

joblib.dump(scaler, scaler_path)
joblib.dump(pca, pca_path)
joblib.dump(svc, svc_path)
kernel_config_out_path.write_text(
    json.dumps(kernel_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Actualizar manifest con entrada de historial de reentrenamiento
history_entry = {
    "retrained_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_version": model_version,
    "dataset_version": dataset_version,
    "test_accuracy": accuracy,
    "test_f1": f1,
    "smoke_test_passed": True,
    "artifact_paths": {
        "scaler": scaler_path.name,
        "pca": pca_path.name,
        "svc": svc_path.name,
        "kernel_config": kernel_config_out_path.name,
    },
}
manifest.setdefault("retrain_history", []).append(history_entry)
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Mostrar rutas guardadas
print("Artefactos guardados:")
for p in (scaler_path, pca_path, svc_path, kernel_config_out_path):
    print(f"  {p}")
print(f"Manifest actualizado: {manifest_path}")
print(f"Versión del modelo: {model_version}")

## 8. Verificación

In [ ]:
# Recargar artefactos desde disco
scaler_loaded = joblib.load(scaler_path)
pca_loaded = joblib.load(pca_path)
svc_loaded = joblib.load(svc_path)
kernel_config_loaded = json.loads(kernel_config_out_path.read_text(encoding="utf-8"))

# Verificar integridad básica
assert kernel_config_loaded == kernel_config, "El kernel_config recargado no coincide"

manifest_reloaded = load_manifest(manifest_path)
last_entry = manifest_reloaded["retrain_history"][-1]
assert last_entry["model_version"] == model_version, "model_version no coincide en el manifest"

print(f"Todos los artefactos verificados correctamente. Versión: {model_version}")